## Extracción del listado de IDs para cada categoría


### Accedo a todos los IDs y luego, con una lista de categorías no comestibles, acoto este listado para evitar perder tiempo en extraer datos que no interesan para el análisis.

In [ ]:
import requests

url_menu = "https://tienda.mercadona.es/api/categories/"
headers_mercadona = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

lista_subgrupos_validos = []

try:
    response = requests.get(url_menu, headers=headers_mercadona, timeout=15)
    if response.status_code == 200:
        data_menu = response.json()

        
        for subcategory in data_menu.get("results", []):
            # se omite el id_general (Nivel 1) ya que esa URL no contiene productos directos

            
            for item in subcategory.get("categories", []):
                id_subgrupo = item.get("id")
                if id_subgrupo:
                    lista_subgrupos_validos.append(str(id_subgrupo))

        print(f"IDs de subgrupos encontrados inicialmente: {len(lista_subgrupos_validos)}")

    else:
        print(f"Error al conectar con la API: Código {response.status_code}")

except Exception as e:
    print(f"Error al extraer los IDs: {e}")

#  IDs a eliminar (Cosméticos, perfumería, etc.)
lista_eliminar = ['201','202','203','199','192','189','185','191','188','187','186','190','194','196','198','213','214','206','207','208','210','212','221','222','225','226','229',
'237','241','234','235','233','231','230','232','229','243','238','239','244','164','166','181','174','168','170','173','171','169']


lista_final_comestibles = [x for x in lista_subgrupos_validos if x not in lista_eliminar]
print(f"Muestra de los primeros 10 IDs válidos: {lista_final_comestibles[:10]}")

## Extracción de IDs de cada categoría

### Iteración entre los IDs de cada categoría extraída en el paso anterior y extracción de los IDs individuales de cada producto.

In [ ]:
import requests


lista_ids_finales= []

for id_categoria in lista_final_comestibles:
  url_menu = f"https://tienda.mercadona.es/api/categories/{id_categoria}/?lang=es&wh=4480"
 
  try:
      response = requests.get(url_menu, headers=headers_mercadona, timeout=15)
      if response.status_code == 200:
          data_menu = response.json()

          
          for subcategory in data_menu.get("categories", []):
              for item in subcategory.get("products", []):
                  id_producto = item.get("id")
                  if id_producto:
                      lista_ids_finales.append(str(id_producto))

          
      else:
          print(
              f"Error al conectar con el catálogo general: Código {response.status_code}"
          )

  except Exception as e:
      print(f"Error al extraer los IDs: {e}")

lista_final_comestibles = list(set(lista_ids_finales))

print("\n Extraction Exitosa ")
print(f"Total de IDs extraídos (con duplicados): {len(lista_ids_finales)}")
print(f"Total de productos únicos listos para procesar: {len(lista_final_comestibles)}")
print(f"Muestra de productos únicos: {lista_final_comestibles[:10]}")

## Extracción y enriquecimiento de datos por producto

### Proceso de consulta y consolidación:
* **Extracción de Mercadona:** Se obtiene la información base de cada producto mediante su ID. 
* **Enriquecimiento con Open Food Facts:** Usando el código de barras (EAN) obtenido de Mercadona, se realiza una segunda consulta para integrar los valores nutricionales (calorías, macros, Nutri-Score y grado NOVA).
* **Manejo de errores y persistencia:** La función implementa un sistema de doble intento y retardos aleatorios (*delays*) para evitar bloqueos por rate limiting. Además, realiza un guardado incremental en un archivo CSV para prevenir pérdidas de datos ante interrupciones.
* **Resultado:** Organiza las variables recolectadas de ambas API en una estructura limpia para generar el DataFrame final de análisis.

In [ ]:
import requests
import time
import os
from IPython.display import display
import random
import pandas as pd

headers_mercadona = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

headers_off = {
    "User-Agent": "ProyectoAnalisis/1.0"
}

def extraer_datos_productos(lista_ids, csv_filename):
    products_list = []
    total_procesados = 0
    exitos_mercadona = 0
    exitos_off = 0

    print("=== INICIANDO EXTRACCIÓN MASIVA ===")

    for id_prod in lista_ids:
        total_procesados += 1
        try:
            url_mercadona = f"https://tienda.mercadona.es/api/products/{id_prod}/"
            res_merca = None

            for intento in range(2):
                try:
                    res_merca = requests.get(url_mercadona, headers=headers_mercadona, timeout=15)
                    if res_merca.status_code == 200:
                        break
                except Exception:
                    pass
                if intento == 0:
                    time.sleep(random.uniform(2.5, 5.5))

            if not res_merca or res_merca.status_code != 200:
                continue

            exitos_mercadona += 1
            data_m = res_merca.json()

            ean_real = data_m.get("ean")
            nombre = data_m.get("display_name")
            price_info = data_m.get("price_instructions", {})

            categoria = "Sin Categoría"
            cat_node = data_m.get("categories", [])
            while isinstance(cat_node, list) and len(cat_node) > 0:
                categoria = cat_node[0].get("name", categoria)
                cat_node = cat_node[0].get("categories", [])

            kcal_100g = proteinas_100g = fibra_100g = nutri_score = None
            procesado = grasa_saturada_100g = azucar_100g = sal_100g = None

            if ean_real:
                url_off = f"https://world.openfoodfacts.org/api/v0/product/{ean_real}.json"

                for intento_off in range(2):
                    try:
                        res_off = requests.get(url_off, headers=headers_off, timeout=15)
                        if res_off.status_code == 200:
                            data_o = res_off.json()
                            
                            if data_o.get("status") == 1:
                                product_data = data_o.get("product", {})
                                nutriments = product_data.get("nutriments", {})

                                kcal_100g = nutriments.get("energy-kcal_100g")
                                proteinas_100g = nutriments.get("proteins_100g")
                                fibra_100g = nutriments.get("fiber_100g")
                                grasa_saturada_100g = nutriments.get("saturated-fat_100g")
                                azucar_100g = nutriments.get("sugars_100g")
                                sal_100g = nutriments.get("salt_100g")
                                nutri_score = product_data.get("nutriscore_grade")
                                procesado = product_data.get("nova_group")
                                
                                exitos_off += 1
                                break
                    except Exception:
                        pass
                    if intento_off == 0:
                        time.sleep(5)

            current_product = {
                "id_producto": id_prod,
                "ean": ean_real,
                "nombre": nombre,
                "categoria": categoria,
                "marca": data_m.get("brand"),
                "precio_empaque": price_info.get("unit_price"),
                "precio_por_kilo": price_info.get("bulk_price"),
                "calorias_100g": kcal_100g,
                "proteinas_100g": proteinas_100g,
                "fibra_100g": fibra_100g,
                "grasa_saturada_100g": grasa_saturada_100g,
                "azucar_100g": azucar_100g,
                "sal_100g": sal_100g,
                "nutri_score": nutri_score,
                "procesado": procesado
            }

            products_list.append(current_product)

            df_single = pd.DataFrame([current_product])
            header_needed = not os.path.exists(csv_filename)
            df_single.to_csv(
                csv_filename,
                mode='a',
                index=False,
                header=header_needed,
                sep=';',
                encoding="utf-8-sig"
            )

            if total_procesados % 50 == 0:
                print(f"Progreso: {total_procesados} IDs evaluados...")

            time.sleep(random.uniform(2.5, 5.5))

        except Exception as e:
            print(f"Error en ID {id_prod}: {e}")

    df_proyecto = pd.DataFrame(products_list)

    print("\n=== EXTRACCIÓN COMPLETADA ===")
    print(f"Total IDs evaluados: {total_procesados}")
    print(f"Productos obtenidos de Mercadona: {exitos_mercadona}")
    print(f"Productos enriquecidos con Open Food Facts: {exitos_off}")

    display(df_proyecto.head(10))

csv_filename_1 = "productos_mercadona_openfoodfacts.csv"
extraer_datos_productos(lista_final_comestibles, csv_filename_1)

## Identificación de registros sin información nutricional

### Análisis de nulos y filtrado por categorías de interés:
* **Detección de valores faltantes:** Se cargan los datos del CSV y se agrupan por categoría para analizar cuáles presentan mayor cantidad de valores nulos en el campo `calorias_100g`.
* **Depuración de categorías:** Se genera un listado de categorías a excluir (no comestibles o irrelevantes) para acotar el enfoque del análisis.
* **Aislamiento de identificadores:** Se extrae un nuevo listado de IDs correspondientes a productos que, habiendo sido extraídos con éxito de Mercadona, carecen de datos nutricionales en Open Food Facts y pertenecen a las categorías seleccionadas para su posterior reintento.

In [ ]:

import pandas as pd
from google.colab import drive


# Con esto cargo los datos del CSV (previamente guardado con separación) para evitar errores y separar luego correctamente.
# Y luego filtro la lista para visualizar qué categorías presentaban faltas en el dato de calorías; con esto puedo hacer un filtro de las categorías que no quiero volver a incluir en el lanzamiento del script de nuevo.


drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/resultado_final.csv',sep=';')
df_filtrado_faltan_calorias = df[df["calorias_100g"].isna()]
df_filtrado_categoria = df_filtrado_faltan_calorias.groupby("categoria")[["id_producto"]].count()
display(df_filtrado_categoria)

# Como tengo el problema de que ahora son más categorías de las que pensaba, lo convierto en una lista para visualizar y, posteriormente, filtrar de mejor manera.

lista_categorias = df_filtrado_categoria.reset_index()["categoria"].tolist()
print(lista_categorias)

categorias_a_eliminar = [
    'Accesorios', 'Arreglos', 'Biberón', 'Braguita y otros', 'Champú y jabón', 'Chupete',
    'Colonia', 'Decoración', 'Pañal talla de 0 a 3', 'Pañal talla de 4 a XL', 'Toallitas', 'Velas',
    'Calabacín y pimiento', 'Cebolla y ajo', 'Carne', 'Cerdo', 'Conejo', 'Cordero', 'Corvina',
    'Cítricos', 'Dorada', 'Ensalada preparada', 'Fruta tropical', 'Lechuga', 'Limón', 'Lubina',
    'Manzana y pera', 'Marisco', 'Marisco de concha', 'Marisco de concha y otros', 'Melón y sandía',
    'Naranja', 'Otras frutas', 'Otras verduras y hortalizas', 'Patata', 'Pepino y zanahoria',
    'Pescado', 'Plátano y uva', 'Pollo', 'Repollo y col', 'Rodaballo', 'Salmón', 'Sardina',
    'Setas y champiñones', 'Tomate', 'Trucha', 'Vacuno', 'Verdura', 'Verduras al vapor','Bacalao','Agua con gas', 'Agua sin gas', 'Barra de pan',
    'Bollería dulce', 'Bollería salada', 'Platos calientes', 'Platos fríos','Agua con gas', 'Agua sin gas','Chicles','Caramelos','Hielo','Otros',
    'Colorante y pimentón','Hierbas','Hierbas aromáticas','Otras especias','Pimienta','Sazonadores','Sal y bicarbonato','Edulcorante y otros'
    ,'Ahumados','Boquerón','Salazones','Café molido','Café soluble','Cápsulas compatibles Dolce gusto','Cápsulas compatibles Nespresso',
    'Cápsulas compatibles Tassimo','Sepia, pulpo y calamar', 'Infusiones', 'Vinagre y otros aderezos', 'Levadura y preparado repostería', 'Caldo en pastillas',
    'Caldo líquido']

nueva_lista_filtrada = [cat for cat in lista_categorias if cat not in categorias_a_eliminar]
print(nueva_lista_filtrada)

# Ahora, con esto voy al DataFrame creado con el CSV original después de la primera extracción de datos y hago una consulta sobre este.
# El objetivo es obtener todos los IDs de productos que fallaron en la primera extracción (sin datos nutricionales) para realizar otro intento, conseguir la mayor cantidad de datos posible y mejorar la calidad del análisis.


ids = df[(df["calorias_100g"].isna()) & (df["categoria"].isin(nueva_lista_filtrada))]["id_producto"].astype(int).tolist()
print(ids)



## Reutilización de la función de extracción para reintentos

### Se ejecuta nuevamente la función de extracción utilizando el listado de IDs filtrados en el paso anterior. El objetivo es realizar un segundo intento de consulta en la API para aquellos productos que inicialmente no registraron datos nutricionales, maximizando así la cobertura del dataset.

In [ ]:
csv_filename = "/content/drive/MyDrive/productos_mercadona_reintentos.csv"
extraer_datos_productos(ids, csv_filename)

##  Unión del primer CSV con los datos recolectados en el segundo

### Se realiza la incorporación de los datos obtenidos en el segundo lanzamiento de la función para generar el CSV definitivo. Este archivo consolidado pasará posteriormente a la etapa de limpieza y procesamiento de datos.

In [ ]:
import pandas as pd


df_principal = pd.read_csv('/content/drive/MyDrive/resultado_final.csv', sep=';')
df_actualizaciones = pd.read_csv('/content/drive/MyDrive/productos_mercadona_reintentos.csv',sep=';')

# Merge por id_producto
df = df_principal.merge(
    df_actualizaciones,
    on='id_producto',
    how='left',
    suffixes=('', '_nuevo')
)

# Rellenar todas las columnas comunes
for col in df_principal.columns:
    if col != 'id_producto' and f'{col}_nuevo' in df.columns:
        df[col] = df[col].fillna(df[f'{col}_nuevo'])
        df.drop(columns=[f'{col}_nuevo'], inplace=True)


df.to_csv(
    '/content/drive/MyDrive/resultado_final_actualizado_completo.csv',
    sep=';',
    index=False
)

print("Archivo guardado: resultado_final_actualizado_completo.csv")

